In [18]:
!python -V

Python 3.11.16


In [19]:
import pandas as pd

In [20]:
import numpy as np

In [21]:
import pickle

In [22]:
import seaborn as sns
import matplotlib.pyplot as plt

In [23]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error

In [24]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1788898117863, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788898117863, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [25]:
import os
print(os.getcwd())
print(mlflow.get_tracking_uri())

/workspaces/mlops-zoomcamp/02-experiment-tracking
sqlite:///mlflow.db


In [26]:
def read_dataframe(filename):
    df = pd.read_csv(filename)

    #On parse les colonne lpep_dropff_date_time. Note : df.lpep_dropoff_datetime est équivalent à faire df['lpep_dropoff_datetime']
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    #On créé un nouvelle colonne 'duration'
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    #Convert city zones identifier (ex:130,65) into text instead of float/integers, otherwise oython will say 130 is greater than 65, which makes no sense
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [27]:
df_train = read_dataframe('./data/green_tripdata_2021-01.csv')
df_val = read_dataframe('./data/green_tripdata_2021-02.csv')

/tmp/ipykernel_25645/1374236984.py:2: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


In [28]:
df_train.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration
0,2.0,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.5,...,0.5,0.00,0.0,NaN,0.3,6.80,2.0,1.0,0.00,3.933333
1,2.0,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.0,...,0.5,2.81,0.0,NaN,0.3,16.86,1.0,1.0,2.75,8.750000
2,2.0,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.0,...,0.5,1.00,0.0,NaN,0.3,8.30,1.0,1.0,0.00,5.966667
3,2.0,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.0,...,0.5,0.00,0.0,NaN,0.3,9.30,2.0,1.0,0.00,7.083333
7,2.0,2021-01-01 00:26:31,2021-01-01 00:28:50,N,1.0,75,75,6.0,0.45,3.5,...,0.5,0.96,0.0,NaN,0.3,5.76,1.0,1.0,0.00,2.316667


In [29]:
len(df_train), len(df_val)

(73908, 61921)

In [30]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

# PU et DO sont des identifiant de zones administratives (zone 130 = Upper East Side, zone 85  = Greenwich Village, zone 41  = JFK Airport) --> ce sont des valeurs discrètes, poas continues

In [31]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer() #DictVectoriser est une classe. Ici on initilise l'objet dv de class dictvectorizer

# Ci dessous : on prends les 2 colonnes categorical et numerical (PU_DO et trip_distace) des dataframe. On les transforme en liste de dictionnaire [{'PU_DO':13_42 , 'trip_distance':34},{'PU_DO':13_42 , 'trip_distance':34},... ] car c'est ce que Dictvectorizer attend
# X_train = dv.fit_transform(train_dicts) --> crée une matrice one hot encoding : n+1 colonnes, avec n, le nombre de valeur de PU_DO différente, la dernière colonne étant 'trip disance'. La dernière colonne contient la valeur de trip distance, tandis que els n colonne contienne 0 ou 1 (1 si la PU_DO coorrespondant à cette valeur de trip distance, 0 sinon)
#X_val et X train doivent contenir les même colonnes (comme pour le cours de Andrew NG., avec les sequencial model, on doit avoir les même mots dans le dictionnaire) --> on utilie la méthode .fit_transform pour X_train, et ensuite .transform pour X_val --> ça assure que X_val a les même colonne que X train
# X_train et X_val ont exactement les même n+1 colonne (les même n colonnes correspondant aux valeurs de PU_D0). Mais comme, X°val a moins d'exemple, il aura des lignes avec uniquement des 0 !!
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [32]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [33]:
# j'ai ajouté ce code pour convertir les int64 en int32 sinon ça faisait un bug
def to_int32(X):
    X = X.tocsr()
    X.indices = X.indices.astype(np.int32)
    X.indptr = X.indptr.astype(np.int32)
    return X

X_train = to_int32(X_train)
X_val = to_int32(X_val)

In [34]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

#mean_squared_error(y_val, y_pred, squared=False) #ne foinctionne pas avec scikitlearn 1.4
np.sqrt(mean_squared_error(y_val, y_pred))

np.float64(7.75871520603533)

In [35]:
#On uvre (ou crée) un fichier nommé lin_reg.bin dans le dossier models/. 
# 'wb' → mode d'ouverture : write (écriture) + binary (binaire), car pickle écrit des octets, pas du texte
# with ... as f_out → ouvre le fichier et le nomme f_out. Le with garantit que le fichier sera automatiquement fermé à la fin du bloc, même si une erreur survient
#(dv, lr) → on sauvegarde un tuple contenant deux objets :
#dv : le DictVectorizer (avec son vocabulaire appris)
#lr : le modèle LinearRegression (avec ses coefficients appris)
#f_out → on écrit dans le fichier ouvert juste avant

with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

**Correction 1 : ce n'est pas une régression linéaire simple****

python
lr = Lasso(alpha)

C'est un Lasso, pas une LinearRegression. C'est une régression linéaire avec régularisation L1 : elle pénalise les gros coefficients, ce qui force certains à devenir exactement 0. Utile ici car avec le one-hot encoding on a des milliers de colonnes.

Le nom de variable lr est trompeur (héritage du copier-coller de la cellule précédente), mais c'est bien un modèle différent.

alpha est l'hyperparamètre qui contrôle la force de cette pénalité — d'où l'intérêt de le logger : tu pourras comparer alpha=0.1, alpha=0.01, etc.

**Petite nuance sur rmse**

Tu dis "fonction de coût". Attention, ce sont deux choses distinctes ici :

La fonction de coût optimisée pendant l'entraînement = MSE + pénalité L1
Le RMSE = la métrique d'évaluation que tu regardes pour juger le modèle

Ça se ressemble mais ce n'est pas identique.

In [36]:

#To give a name to the run, you can follow these 2 methods
#with mlflow.start_run(run_name="lasso-alpha-0.1"):
   # mlflow.set_tag("developer", "cristian")
    #...
#OR
#with mlflow.start_run():
#    mlflow.set_tag("mlflow.runName", "lasso-alpha-0.1")


with mlflow.start_run():

    

    mlflow.set_tag("developer", "cristian")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")

    #C'est pas une régression linéaire mais un Lasso, pas une LinearRegression. C'est une régression linéaire avec régularisation L1
    #alpha est ici le paramètr ed erégularsation (le lambda de Andrew Ng., pas le learning rate)
    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    #rmse = mean_squared_error(y_val, y_pred, squared=False)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [37]:
print("tracking uri :", mlflow.get_tracking_uri())

run = mlflow.last_active_run()
print("run_id       :", run.info.run_id)
print("experiment_id:", run.info.experiment_id)
print("status       :", run.info.status)
print("artifact_uri :", run.info.artifact_uri)

tracking uri : sqlite:///mlflow.db
run_id       : 30cfa6db5eb84c5891fcb83dd6ee4cff
experiment_id: 1
status       : FINISHED
artifact_uri : /workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1/30cfa6db5eb84c5891fcb83dd6ee4cff/artifacts


In [38]:
Lasso?

Init signature:
Lasso(
    alpha=1.0,
    *,
    fit_intercept=True,
    precompute=False,
    copy_X=True,
    max_iter=1000,
    tol=0.0001,
    warm_start=False,
    positive=False,
    random_state=None,
    selection='cyclic',
)
Docstring:     
Linear Model trained with L1 prior as regularizer (aka the Lasso).

The optimization objective for Lasso is::

    (1 / (2 * n_samples)) * ||y - Xw||^2_2 + alpha * ||w||_1

Technically the Lasso model is optimizing the same objective function as
the Elastic Net with ``l1_ratio=1.0`` (no L2 penalty).

Read more in the :ref:`User Guide <lasso>`.

Parameters
----------
alpha : float, default=1.0
    Constant that multiplies the L1 term, controlling regularization
    strength. `alpha` must be a non-negative float i.e. in `[0, inf)`.

    When `alpha = 0`, the objective is equivalent to ordinary least
    squares, solved by the :class:`LinearRegression` object. For numerical
    reasons, using `alpha = 0` with the `Lasso` object is not advised.

In [39]:
import xgboost as xgb

In [40]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [41]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [42]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

**fmin — la recherche d'hyperparamètres**

fmin vient de la librairie hyperopt. Son nom signifie "function minimize" : elle cherche les hyperparamètres qui minimisent la valeur retournée par ta fonction.

Correction importante : fmin ne cherche pas "le run où l'erreur est minimale" dans MLflow. Elle ne connaît même pas l'existence de MLflow. Elle est complètement indépendante — MLflow enregistre juste en parallèle, comme un observateur.

Oui, fn=objective passe bien la fonction elle-même, sans parenthèses. Tu ne l'appelles pas, tu la donnes à fmin, qui va l'appeler 50 fois avec des hyperparamètres différents.

Voici la boucle qui se déroule :

fmin tire un jeu d'hyperparamètres du search_space
   ↓
appelle objective(params)
   ↓
objective entraîne XGBoost, calcule le rmse, log dans MLflow
   ↓
objective retourne {'loss': rmse, 'status': STATUS_OK}
   ↓
fmin lit le 'loss' et en déduit où chercher ensuite
   ↓
(répète 50 fois)

C'est pour ça que le return à la fin de objective est obligatoire : c'est le contrat entre ta fonction et fmin. La clé doit s'appeler exactement 'loss', sinon hyperopt ne saura pas quoi minimiser.

Argument	Rôle
fn=objective	La fonction à minimiser
space=search_space	Où chercher
algo=tpe.suggest	Comment chercher (voir ci-dessous)
max_evals=50	Nombre d'essais
trials=Trials()	Objet qui mémorise l'historique des essais
Le point le plus intéressant : tpe.suggest

C'est ce qui distingue hyperopt d'une simple grid search.

Grid search : teste toutes les combinaisons d'une grille → explosion combinatoire
Random search : tire au hasard → n'apprend rien des essais passés
TPE (Tree-structured Parzen Estimator) : apprend des essais précédents

Le TPE construit un modèle probabiliste du genre "les bons résultats avaient un learning_rate autour de 0.1 et un max_depth plutôt faible", puis concentre ses tirages suivants dans ces zones prometteuses. C'est de l'optimisation bayésienne.

D'où l'intérêt de l'objet Trials() : il garde l'historique dont le TPE a besoin pour raisonner.

⚠️ Attention à la confusion de vocabulaire ! Ici objective n'a rien à voir avec ta fonction Python objective. C'est un hyperparamètre XGBoost qui désigne la fonction de perte à optimiser (régression). Malheureux hasard de nommage dans le cours.

Par ailleurs 'reg:linear' est obsolète. Chez toi il faut mettre :

python
'objective': 'reg:squarederror'
Et comme d'habitude :

python
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

**Pourquoi on met la fonction de perte en paramètres ?** 

D'abord, regarde bien le search_space
python
search_space = {
    'max_depth': scope.int(hp.quniform(...)),      # ← varie
    'learning_rate': hp.loguniform(...),           # ← varie
    'reg_alpha': hp.loguniform(...),               # ← varie
    'reg_lambda': hp.loguniform(...),              # ← varie
    'min_child_weight': hp.loguniform(...),        # ← varie
    'objective': 'reg:linear',                     # ← constante !
    'seed': 42                                     # ← constante !
}

Les cinq premiers utilisent hp.* → ce sont des distributions, hyperopt va y piocher.
Les deux derniers sont des valeurs fixes → hyperopt les recopie telles quelles à chaque essai.

Donc objective n'est pas optimisé. Il ne varie jamais sur les 50 runs.

Alors pourquoi est-il dans ce dictionnaire ?

Pour une raison purement technique. Regarde le chemin que prend ce dictionnaire :

search_space  →  fmin  →  objective(params)  →  xgb.train(params=params, ...)

xgb.train attend un seul dictionnaire contenant toute la configuration du modèle. L'API native de XGBoost ne fait aucune distinction entre "ce qui se règle" et "ce qui est structurel" — tout passe par params.

Donc objective et seed sont dans le search_space non pas parce qu'on veut les optimiser, mais parce que c'est le seul véhicule pour les faire arriver jusqu'à xgb.train.

C'est un peu bancal comme design, tu as raison de tiquer.

Comparaison avec ce que tu connais
	Choix de la loss	Réglage
sklearn	Encodé dans la classe : LinearRegression, Lasso, Ridge	Arguments du constructeur
XGBoost natif	Une clé du dict params	Les autres clés du même dict

En sklearn, changer de fonction de perte = changer de classe. En XGBoost, c'est juste une chaîne de caractères, parce qu'un seul algorithme (le boosting d'arbres) sait gérer régression, classification et ranking. La loss est ce qui détermine laquelle des trois tu fais.

Ceci dit, la nuance mérite d'être posée

objective peut légitimement être considéré comme un hyperparamètre au sens strict : tout ce qui est fixé avant l'entraînement et non appris à partir des données en est un.

Et en pratique, on le fait parfois varier ! Par exemple pour une régression :

python
'reg:squarederror'      # pénalise fortement les grosses erreurs
'reg:absoluteerror'     # plus robuste aux valeurs aberrantes
'reg:pseudohubererror'  # compromis entre les deux

Sur tes durées de trajets, tu pourrais comparer ces trois-là et voir lequel donne le meilleur résultat. Ce serait un choix de modélisation légitime.

Mais c'est un choix de nature différente de celui du learning rate :

learning_rate → question d'optimisation : à quelle vitesse converger
objective → question de modélisation : qu'est-ce qu'une bonne prédiction, comment je veux pénaliser mes erreurs

Ta gêne vient de là, et elle est fondée. On mélange dans un même dictionnaire des paramètres qui n'ont pas le même statut conceptuel.

In [45]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 20, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    #'objective': 'reg:linear',
    'objective': 'reg:squarederror',
    'seed': 42
}

#fmin vient de la librairie hyperopt. Son nom signifie "function minimize" : elle cherche les hyperparamètres qui minimisent la valeur retournée par ta fonction.

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

[0]	validation-rmse:8.82046                           
[1]	validation-rmse:7.49757                           
[2]	validation-rmse:7.02508                           
[3]	validation-rmse:6.84703                           
[4]	validation-rmse:6.76701                           
[5]	validation-rmse:6.73237                           
[6]	validation-rmse:6.71305                           
[7]	validation-rmse:6.70408                           
[8]	validation-rmse:6.69932                           
[9]	validation-rmse:6.69670                           
[10]	validation-rmse:6.69400                          
[11]	validation-rmse:6.68918                          
[12]	validation-rmse:6.67956                          
[13]	validation-rmse:6.67555                          
[14]	validation-rmse:6.67143                          
[15]	validation-rmse:6.66719                          
[16]	validation-rmse:6.66418                          
[17]	validation-rmse:6.66089                          
[18]	valid

job exception: got an unexpected keyword argument 'squared'



  0%|          | 0/10 [01:39<?, ?trial/s, best loss=?]


TypeError: got an unexpected keyword argument 'squared'

In [18]:
mlflow.xgboost.autolog(disable=True)

In [ ]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    #rmse = mean_squared_error(y_val, y_pred, squared=False)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

NameError: name 'mlflow' is not defined

In [23]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        mlflow.log_metric("rmse", rmse)
        

/Users/cristian.martinez/miniconda3/envs/exp-tracking-env/lib/python3.9/site-packages/sklearn/svm/_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
